In [ ]:
# =========================
# 1) Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os
import librosa
import soundfile as sf
import numpy as np
from scipy.signal import butter, lfilter
from tqdm import tqdm

# =========================
# Paths
# =========================
input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/MusicGen_Output/"
raw48_dir = "/content/drive/MyDrive/MusicGen_Project/Data/FlashSR_Raw48"
lp8_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Degraded/FlashSR/"

os.makedirs(raw48_dir, exist_ok=True)
os.makedirs(lp8_dir, exist_ok=True)

# =========================
# Low-pass filter
# =========================
def lowpass_filter(audio, sr, cutoff=8000, order=5):
    nyquist = 0.5 * sr
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low')
    return lfilter(b, a, audio)

target_sr = 48000
target_len = int(5.12 * target_sr)  # 245760 samples

# =========================
# Loop over all files
# =========================
for file in tqdm(os.listdir(input_dir)):
    if file.endswith(".wav"):
        path = os.path.join(input_dir, file)

        y, sr = librosa.load(path, sr=None)

        # Resample to 48k
        y_48 = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

        # Trim/pad to 5.12 sec
        if len(y_48) > target_len:
            y_48 = y_48[:target_len]
        else:
            y_48 = np.pad(y_48, (0, target_len - len(y_48)))

        # Save raw 48k
        sf.write(os.path.join(raw48_dir, file), y_48, target_sr)

        # Low-pass 8k
        y_lp8 = lowpass_filter(y_48, target_sr, cutoff=8000)

        # Save 8k input
        sf.write(os.path.join(lp8_dir, file), y_lp8, target_sr)

print("Done. Raw48 and 8k input ready.")

100%|██████████| 30/30 [00:01<00:00, 21.26it/s]

Done. Raw48 and 8k input ready.


In [ ]:
# =========================
# Clone repo
# =========================
!git clone https://github.com/jakeoneijk/FlashSR_Inference.git
%cd /content/FlashSR_Inference

Cloning into 'FlashSR_Inference'...
remote: Enumerating objects: 280, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (239/239), done.
remote: Total 280 (delta 31), reused 269 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (280/280), 16.66 MiB | 8.13 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/FlashSR_Inference


In [ ]:
# =========================
# Install dependencies
# =========================
!pip install -U pip setuptools wheel
!pip install torch torchaudio torchvision
!pip install -e .
!pip install librosa soundfile numpy scipy einops tqdm omegaconf pyyaml

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# =========================
# Checkpoint paths
# =========================
student_ldm_ckpt_path = "/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/student_ldm.pth"
sr_vocoder_ckpt_path  = "/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/sr_vocoder.pth"
vae_ckpt_path         = "/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/vae.pth"

In [ ]:
# =========================
# Sanity check files exist
# =========================
import os

for p in [student_ldm_ckpt_path, sr_vocoder_ckpt_path, vae_ckpt_path]:
    print(p, "->", os.path.exists(p))

/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/student_ldm.pth -> True
/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/sr_vocoder.pth -> True
/content/drive/MyDrive/MusicGen_Project/FlashSR_Pmodel/vae.pth -> True


In [1]:
# =========================
# Imports + device
# =========================
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
# =========================
# Load FlashSR
# =========================
from FlashSR.FlashSR import FlashSR

flashsr = FlashSR(
    student_ldm_ckpt_path,
    sr_vocoder_ckpt_path,
    vae_ckpt_path
).to(device)

flashsr.eval()
print("FlashSR loaded.")

There is no Hparams


/content/FlashSR_Inference/TorchJaekwon/Model/Diffusion/External/diffusers/configuration_utils.py:53: SyntaxWarning: invalid escape sequence '\.'
  _re_configuration_file = re.compile(r"config\.(.*)\.json")
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of

FlashSR loaded.


In [ ]:
# =========================
# Batch FlashSR on entire folder
# =========================
import os
from tqdm import tqdm
import torchaudio
import torch
import torch.nn.functional as F
import soundfile as sf

input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Degraded/FlashSR/"
output_dir = "/content/drive/MyDrive/MusicGen_Project/Data/FlashSR_Output"
os.makedirs(output_dir, exist_ok=True)

TARGET_SR = 48000
TARGET_LEN = 245760  # 5.12 sec

def load_audio_fixed(path, target_sr=48000, target_len=245760):
    audio, sr = torchaudio.load(path)   # [C, T]

    # Resample to 48k if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        audio = resampler(audio)

    # Trim or pad to 5.12 sec
    cur_len = audio.shape[1]
    if cur_len > target_len:
        audio = audio[:, :target_len]
    elif cur_len < target_len:
        audio = F.pad(audio, (0, target_len - cur_len))

    return audio

wav_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".wav")]
wav_files.sort()

print("Found", len(wav_files), "files")

for fname in tqdm(wav_files):
    in_path = os.path.join(input_dir, fname)
    out_path = os.path.join(output_dir, fname)

    try:
        audio = load_audio_fixed(in_path, TARGET_SR, TARGET_LEN).to(device)

        with torch.no_grad():
            pred_hr_audio = flashsr(audio, lowpass_input=True)  # change to False to test

        pred_hr_audio = pred_hr_audio.detach().cpu()

        # Save output
        if pred_hr_audio.ndim == 3:
            arr = pred_hr_audio.squeeze(0).T.numpy()
        else:
            arr = pred_hr_audio.T.numpy()

        sf.write(out_path, arr, TARGET_SR)

    except Exception as e:
        print(f"Error on {fname}: {e}")

print("All files processed.")

Found 30 files


100%|██████████| 30/30 [01:15<00:00,  2.51s/it]

All files processed.


In [ ]:
# =========================
# Listen BEFORE vs AFTER for all files
# =========================
import os
import glob
from IPython.display import Audio, display

FlashSR_input_dir = input_dir
FlashSR_output_dir = output_dir

input_files = sorted(glob.glob(os.path.join(FlashSR_input_dir, "*.wav")))

print("Found files:", len(input_files))

for in_file in input_files:
    base = os.path.basename(in_file)
    out_file = os.path.join(
        FlashSR_output_dir,
        base.replace("_FlashSR_input", "_FlashSR_output")
    )

    print("\n==============================")
    print("File:", base)
    print("==============================")

    if os.path.exists(out_file):
        print("FlashSR INPUT (degraded)")
        display(Audio(in_file))

        print("FlashSR OUTPUT (restored)")
        display(Audio(out_file))
    else:
        print("Output file not found for:", base)

Output hidden; open in https://colab.research.google.com to view.